# 3일차 파트 2 실습 풀이

`3일차_02_실습.ipynb`의 풀이와 해설입니다. 풀이는 하나의 예시이며, 테스트 셀을 통과한다면 다르게 작성해도 됩니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: python -m pip install sentence-transformers scikit-learn pypdf python-docx openpyxl pandas
import numpy as np

import embed

DATA = 'data/3일차'
CORPUS = 'data/2일차/corpus'      # 2일차와 같은 문서 253건
OUT = 'out'                        # 예제가 만드는 파일을 두는 폴더
os.makedirs(OUT, exist_ok=True)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 준비

아래 셀은 실습에 쓰는 도구를 불러오고 함수를 정의합니다. 그대로 실행합니다. 재료는 `data/3일차/docs_raw_m2/`(원본 문서 10건)와 `data/3일차/docs_raw_m2_ext/`(열리지 않는 파일 2건)입니다. 예제와 다른 문서입니다. 이 실습에는 `pypdf`, `python-docx`, `openpyxl`, `pandas`, `sentence-transformers`와 임베딩 모델이 필요합니다. 설치와 모델 준비는 교재의 3일차 "실습 환경 준비"를 따릅니다. 위의 설치 명령에 없는 패키지는 그곳의 명령으로 설치합니다.

| 이름 | 내용 |
| --- | --- |
| `read_pdf(path)`, `read_docx(path)`, `read_md(path)` | PDF, DOCX, MD 파일의 글을 문자열로 돌려줍니다. MD는 맨 앞의 front matter를 뺀 본문을 돌려줍니다. 열리지 않으면 오류를 발생시킵니다 |
| `chunk_fixed(text, size)` | 글자 수로 자른 청크의 목록을 돌려줍니다 |
| `repeated_lines(line_lists)` | 전체 문서의 80%를 넘는 문서에 나오는 줄을 돌려줍니다. 반복 줄의 후보입니다 |

In [ ]:
# 실습 준비
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from docx import Document
from openpyxl import load_workbook
from pypdf import PdfReader

import checks_d3_2

RAW2 = f'{DATA}/docs_raw_m2'
EXT2 = f'{DATA}/docs_raw_m2_ext'
CONTROL = re.compile(r'[\x00-\x08\x0b-\x1f\x7f]')


def read_pdf(path):
    return '\n'.join(page.extract_text() for page in PdfReader(path).pages)


def read_docx(path):
    return '\n'.join(paragraph.text for paragraph in Document(path).paragraphs)


def read_md(path):
    text = open(path, encoding='utf-8').read()
    if text.startswith('---\n'):                     # front matter는 본문이 아니므로 뺍니다
        text = text[4:].partition('\n---\n')[2]
    return text


def chunk_fixed(text, size=300):
    return [text[start:start + size] for start in range(0, len(text), size) if text[start:start + size].strip()]


def repeated_lines(line_lists, ratio=0.8):
    counter = Counter(line for lines in line_lists for line in set(lines))
    return [line for line, count in counter.items() if count > len(line_lists) * ratio]


files = sorted(path for path in Path(RAW2).rglob('*') if path.is_file())
print(f'{len(files)}건:', [path.name for path in files])

## 실습 2 새 원본 문서를 청크로 만드는 함수 작성하기

**목적** 처음 보는 문서 묶음에서 반복 줄을 찾아 지우고, 엑셀의 장식 줄을 건너뛰고, 열리지 않는 파일을 실패 목록에 남기는 파이프라인 함수를 직접 작성합니다.

**재료** `data/3일차/docs_raw_m2/`의 원본 문서 10건(PDF 4, DOCX 4, XLSX 1, MD 1)과 `docs_raw_m2_ext/`의 파일 2건, 질문 5개(`실습2_질문_5.csv`. 정답 문서를 맞히는 데 씁니다). 예제의 문서와 반복 줄의 종류가 다릅니다.

**할 일**

1. 후보 셀을 실행해 PDF와 DOCX에서 80%를 넘는 문서에 나오는 줄을 봅니다. 그 줄을 눈으로 읽고, 어느 줄이 반복되는 노이즈인지 판단해 `NOISE_WORDS`에 낱말을 적습니다. 후보에 없는 반복 줄이 있는지도 문서의 원문을 읽어 확인합니다.
2. 함수 네 개를 작성합니다.
   - `find_header_row(rows)`: `rows`는 엑셀의 각 줄을 튜플로 만든 목록입니다. 모든 칸이 채워진 첫 줄의 위치(첫 줄이 0)를 돌려줍니다. 그런 줄이 없으면 `None`입니다. 비어 있는 칸은 `None`이나 빈 문자열입니다.
   - `read_xlsx(path)`: 열 이름 줄을 `find_header_row()`로 찾고, 그 아래 줄마다 한 줄 문장(`열 이름 값, 열 이름 값, ...`)을 만들어 줄바꿈으로 이은 문자열을 돌려줍니다. 비어 있는 칸은 문장에 쓰지 않습니다.
   - `drop_noise(lines, words)`: 줄의 목록에서 노이즈를 지우고 `(남긴 줄의 목록, 지운 줄 수)`를 돌려줍니다. 줄에서 공백을 모두 뺀 글에 `words`의 낱말(공백을 뺀 글)이 들어 있거나, 줄 전체가 `- 1 -` 꼴의 쪽번호이면 노이즈입니다. 쪽번호가 `…확인하십시오. -1-`처럼 다른 글 끝에 붙어 나오는 줄은 이 규칙으로는 지울 수 없으므로 낱말로 지웁니다. (참고: 교재 예제 2-2의 `is_noise`, `PAGE_NUMBER`)
   - `process(folder, words, size)`: 폴더 아래의 모든 파일을 열어 청크 목록과 처리 보고를 돌려줍니다. 청크는 `{'doc_id': 파일 이름에서 확장자를 뺀 것, 'text': 청크}`입니다. 처리 보고는 `{'열린 파일': 건수, '지운 줄': 건수, '실패': [(파일 이름, 오류 이름), ...]}`입니다. 파일이 열리지 않으면 실패 목록에 남기고 나머지를 계속 처리합니다. 제어 문자는 준비 셀의 `CONTROL`로 지웁니다. 노이즈 규칙(`drop_noise()`)은 엑셀을 뺀 형식(PDF, DOCX, MD)에 적용합니다. 엑셀의 행 문장은 표의 데이터이므로, 노이즈 낱말과 같은 글이 들어 있어도 지우면 안 됩니다(교재 예제 2-7에서는 열 이름 `문서번호`가 노이즈 낱말과 겹쳐, 규칙을 엑셀에도 적용하면 공정 스펙 30행이 모두 노이즈로 판정됐습니다). 지운 줄 수는 노이즈로 판정한 줄의 수이고, 빈 줄은 세지 않습니다. (참고: 교재 예제 2-7의 `process`)
3. 테스트 셀을 실행합니다.
4. 측정 셀을 실행해 문서 10건을 처리하고 청크 크기 100자와 400자의 recall@1, recall@3을 측정합니다. 나머지 2건(`docs_raw_m2_ext/`)은 처리 보고의 실패 목록으로 확인합니다.

**완료 기준** 테스트 셀이 "실습 2 통과"를 출력합니다. 측정 결과의 숫자와 함께, 이 숫자만으로 청크 크기를 고를 수 있는지 한 줄로 판단을 적습니다.

<details><summary>힌트 1</summary>

PDF의 "기밀" 줄은 `pypdf`로 뽑으면 글자 사이에 공백이 들어 있습니다(`기 밀 · 내 부 …`). `'기밀' in line`은 거짓입니다. 줄과 낱말에서 공백을 모두 뺀 뒤 비교합니다. `''.join(line.split())`이 공백을 모두 뺍니다.
</details>

<details><summary>힌트 2</summary>

`find_header_row()`의 "모든 칸이 채워졌다"는 `all(cell not in (None, '') for cell in row)`입니다. `read_xlsx()`는 `load_workbook(path).active.iter_rows(values_only=True)`로 줄을 읽어 `rows = list(...)`로 만든 뒤 열 이름 줄 아래를 순서대로 처리합니다.
</details>

<details><summary>힌트 3</summary>

`process()`는 `sorted(Path(folder).rglob('*'))`로 모든 파일을 순서대로 보고, `path.suffix.lower()`로 형식을 가립니다. 엑셀이면 `drop_noise()`를 부르지 않고 지운 줄을 0줄로 둡니다. 파일을 여는 코드를 `try`로 감싸 `except Exception as error`에서 `type(error).__name__`을 실패 목록에 적습니다.
</details>

In [ ]:
# 실습 2 (1) 반복 줄의 후보를 봅니다
pdf_lines, docx_lines = [], []
for path in files:
    if path.suffix == '.pdf':
        pdf_lines.append([line.strip() for line in read_pdf(path).split('\n') if line.strip()])
    elif path.suffix == '.docx':
        docx_lines.append([line.strip() for line in read_docx(path).split('\n') if line.strip()])
print('PDF 후보:', repeated_lines(pdf_lines))
print('DOCX 후보:', repeated_lines(docx_lines))
print()
print(read_pdf(f'{RAW2}/pdf/NOTI-031.pdf'))

In [ ]:
# 실습 2 (2) 지울 줄의 낱말을 적습니다
NOISE_WORDS = ['무단 반출 금지', '문서관리시스템', '문서번호', '기밀', '— 계속 —', 'Printed', '출력본']

In [ ]:
# 실습 2 (3) 함수 네 개를 작성합니다
PAGE_NUMBER = re.compile(r'-\s*\d+\s*-')


def find_header_row(rows):
    """모든 칸이 채워진 첫 줄의 위치(첫 줄이 0)를 돌려줍니다. 그런 줄이 없으면 None."""
    for position, row in enumerate(rows):
        if all(cell not in (None, '') for cell in row):
            return position
    return None


def read_xlsx(path):
    """열 이름 줄 아래의 각 줄을 '열 이름 값, 열 이름 값' 한 줄로 만들어 줄바꿈으로 이어 돌려줍니다."""
    rows = list(load_workbook(path).active.iter_rows(values_only=True))
    header_at = find_header_row(rows)
    header = rows[header_at]
    sentences = []
    for row in rows[header_at + 1:]:
        cells = [f'{column} {value}' for column, value in zip(header, row) if value not in (None, '')]
        if cells:
            sentences.append(', '.join(cells))
    return '\n'.join(sentences)


def drop_noise(lines, words):
    """노이즈 줄을 지우고 (남긴 줄의 목록, 지운 줄 수)를 돌려줍니다. 공백을 뺀 글로 비교합니다."""
    squeezed_words = [''.join(word.split()) for word in words]
    kept = []
    for line in lines:
        squeezed = ''.join(line.split())
        if any(word in squeezed for word in squeezed_words) or PAGE_NUMBER.fullmatch(line.strip()):
            continue
        kept.append(line)
    return kept, len(lines) - len(kept)


def process(folder, words, size):
    """폴더 아래의 모든 파일을 열어 (청크 목록, 처리 보고)를 돌려줍니다."""
    chunks, report = [], {'열린 파일': 0, '지운 줄': 0, '실패': []}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        try:
            if suffix == '.pdf':
                text = read_pdf(path)
            elif suffix == '.docx':
                text = read_docx(path)
            elif suffix == '.xlsx':
                text = read_xlsx(path)
            else:
                text = read_md(path)
        except Exception as error:                   # 열리지 않은 파일은 목록에 남기고 계속합니다
            report['실패'].append((path.name, type(error).__name__))
            continue
        lines = [CONTROL.sub('', line).strip() for line in text.split('\n') if line.strip()]
        if suffix == '.xlsx':                         # 엑셀의 행 문장은 표의 데이터이므로 노이즈 규칙을 적용하지 않습니다
            kept, removed = lines, 0
        else:
            kept, removed = drop_noise(lines, words)
        report['열린 파일'] += 1
        report['지운 줄'] += removed
        for piece in chunk_fixed('\n'.join(kept), size):
            chunks.append({'doc_id': path.stem, 'text': piece})
    return chunks, report

In [ ]:
# 실습 2 확인 (임베딩 모델도 API도 부르지 않습니다)
checks_d3_2.check_pipeline(find_header_row, read_xlsx, drop_noise, process)

In [ ]:
# 실습 2 (4) 문서 10건으로 청크 크기별로 측정하고, 열리지 않는 2건의 처리 보고를 봅니다
import numpy as np

import embed

questions = pd.read_csv(f'{DATA}/실습2_질문_5.csv')
for size in [100, 400]:
    chunks, report = process(RAW2, NOISE_WORDS, size)
    vectors = embed.passages([chunk['text'] for chunk in chunks])
    recall1 = recall3 = 0
    for question, gold in zip(questions['질문'], questions['정답']):
        order = np.argsort(-(vectors @ embed.query(question)), kind='stable')
        found = [chunks[position]['doc_id'] for position in order[:3]]
        recall1 += found[0] == gold
        recall3 += gold in found
    print(f'{size}자: 청크 {len(chunks)}건, recall@1 {recall1}/5, recall@3 {recall3}/5')
print(report)
_, report_ext = process(EXT2, NOISE_WORDS, 400)
print(report_ext)

**해설**

아래 숫자는 2026-11-25에 실행한 결과입니다. 모델의 응답이 아니라 검색과 문서 처리 결과이므로 같은 환경에서는 거의 같은 값이 나옵니다.

| 지울 낱말 | 지운 줄 | 100자: 청크 수 | 100자: recall@1, recall@3 | 400자: 청크 수 | 400자: recall@1, recall@3 |
| --- | --- | --- | --- | --- | --- |
| 없음 | 0줄 | 50 | 4/5, 5/5 | 15 | 3/5, 4/5 |
| 예제 2-2의 낱말 3개 | 29줄 | 38 | 4/5, 5/5 | 12 | 3/5, 5/5 |
| 풀이의 낱말 7개 | 45줄 | 33 | 4/5, 5/5 | 10 | 3/5, 4/5 |

- **후보 셀은 반복 줄 일부만 보여 줍니다.** PDF 4건의 후보 8줄에는 바닥글 두 줄(쪽번호가 줄 끝에 붙어 있습니다), "기밀" 줄, "— 계속 —" 줄, "Printed" 줄과 개정 이력 세 줄이 있습니다. 머리말 `SL-FAB 반도체 · 사내 일반 · 무단 반출 금지 문서번호 NOTI-031`은 문서 번호가 문서마다 달라서 후보에 없습니다. 후보 셀이 출력한 `NOTI-031`의 원문을 읽어야 찾을 수 있습니다. DOCX의 후보는 "※ 본 문서는 출력본입니다…" 한 줄입니다. 개정 이력 세 줄은 반복되지만 문서 내용이어서 이 풀이는 지우지 않았습니다. `계속` 같은 일반 낱말을 적으면 규칙이 본문에 같은 낱말이 든 줄도 지웁니다. 그래서 풀이는 줄에 맞는 구절(`— 계속 —`)을 적었습니다.
- **"기밀" 줄은 글자 사이에 공백이 들어 있습니다.** `기 밀  ·  내 부  검 토 용  초 안  ·  배 포  금 지`는 `'기밀' in line`으로 찾을 수 없어서, 줄과 낱말에서 공백을 뺀 글로 비교합니다.
- **쪽번호 규칙만으로는 줄을 지우지 못했습니다.** `pypdf`는 쪽번호를 바닥글 줄 끝에 붙여(`…확인하십시오. -1-`) 내므로, 쪽번호만 있는 줄이 없습니다. 낱말을 비우면 지운 줄이 0줄입니다. 바닥글의 낱말(`문서관리시스템`)을 넣어야 이 줄을 지웁니다.
- **지울수록 청크가 줄고, recall@3은 거의 변하지 않았습니다.** 100자 청크가 50건에서 33건으로 줄었지만 recall@1은 4/5, recall@3은 5/5로 그대로입니다. 반복 줄은 청크 수와 인덱스 크기를 늘립니다. 이 질문 5개에서 검색 순위를 해칠 만큼은 아니었습니다.
- **400자의 recall@3이 5/5에서 4/5로 내려간 것은 질문 한 개입니다.** 지운 줄을 늘린 쪽이 더 나쁜 결과를 냈지만, 노이즈를 더 지워서 검색이 나빠졌다고 해석할 수 없습니다. 질문 5개에서 한 개의 차이는 20%p입니다.
- **100자와 400자도 이 숫자만으로 고를 수 없습니다.** 100자의 recall@1이 4/5, 400자가 3/5로 100자가 높지만 한 문제 차이입니다. 예제 2-5에서 질문 10개와 20개로 측정한 크기의 순서가 바뀐 것과 같은 문제입니다. 크기를 정하려면 질문 세트를 늘려야 합니다.
- **엑셀에는 노이즈 규칙을 적용하지 않습니다.** 테스트가 만드는 엑셀 파일에는 알람명이 `기밀 누설`(챔버나 배관의 기밀이 깨져 공기가 새는 상태)인 행이 있습니다. 노이즈 낱말 `기밀`로 검사하면 규칙이 이 행을 지우고, 오류는 나지 않습니다. 실습 재료의 `설비알람_코드표.xlsx`에는 노이즈 낱말과 겹치는 글이 없어서, 이 규칙이 측정 숫자를 바꾸지는 않습니다.
- **`docs_raw_m2_ext/`의 두 파일은 `process()`가 실패 목록에 남깁니다.** `EHS-022.docx`는 `BadZipFile`, `NOTI-032.pdf`는 `EmptyFileError`입니다. 열린 파일이 0건이어도 오류 없이 끝나므로, 처리 보고를 읽어야 이 사실을 알 수 있습니다.